# Reading -> Transforming -> Writing (from : bronze.erp_loc_a101_raw to silver.erp_loc_a101)

## Reading the data

In [0]:
from pyspark.sql.functions import col, trim, to_date, lit, when
from pyspark.sql.types import StringType

In [0]:
df = spark.table("workspace.bronze.erp_loc_a101_raw")

In [0]:
df.count()

In [0]:
df.printSchema()

In [0]:
df.limit(10).display()

## Validating Data

### Checking for country value

In [0]:
df.select("COUNTRY").distinct().count()

In [0]:
country_groupBy = df.groupBy("COUNTRY").count()
country_groupBy.select("COUNTRY", "count", length(col("COUNTRY")).alias("name_length")).display()

### Checking for duplicate data based on CID

In [0]:
print(f"Total row count:{df.count()}")
df.groupBy("CID").count().filter(col("count") > 1).count()

## Transforming Data

### Changing Column Names

In [0]:
df = df.withColumnRenamed("CNTRY", "COUNTRY")
df.printSchema()

### Trimming Data

In [0]:
for field_name in df.schema.fieldNames():
    df = df.withColumn(field_name, trim(col(field_name)))

df.limit(10).display()

### Making Empty Country To Null And Country Code to Country Name

In [0]:
COUNTRY_CODE_MAP = {
    "US":"United States Of America",
    "USA":"United States Of America",
    "DE":"Germany",
    "United States": "United States Of America",
    "United States Of America": "United States of America"
}

expr = when(length(col("COUNTRY")) == 0, lit(None))
for code, full_name in COUNTRY_CODE_MAP.items():
    expr = expr.when(col("COUNTRY") == code, full_name)

expr = expr.otherwise(col("COUNTRY"))
                      

df = df.withColumn("COUNTRY", expr)


### Setting proper case for country name

In [0]:
from pyspark.sql.functions import initcap

df = df.withColumn("COUNTRY", initcap(col("COUNTRY")))

df.groupBy("COUNTRY").count().display()

## Writing data

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.erp_loc_a101")